In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import re

from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table

In [23]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2025-11-01"

# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

In [ ]:
# BRONZE
# create bronze datalake - lms
bronze_lms_directory = "datamart/bronze/lms/"

if not os.path.exists(bronze_lms_directory):
    os.makedirs(bronze_lms_directory)

# run bronze backfill - lms
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_lms_table(date_str, bronze_lms_directory, spark)



In [ ]:
# bronze - financials and attributes
table_dict = {
    'source': ['data/features_attributes.csv', 'data/features_financials.csv'],
    'directory': ['datamart/bronze/attributes/', 'datamart/bronze/financials/'],
    'filename': ['cust_attr', 'cust_fin']
             }

for i in range(len(table_dict['source'])):
    if not os.path.exists(table_dict['directory'][i]):
        os.makedirs(table_dict['directory'][i])
    utils.data_processing_bronze_table.process_bronze_other_tables(
        table_dict['source'][i], 
        table_dict['directory'][i], 
        table_dict['filename'][i], 
        spark
    )

In [ ]:
import importlib
import utils.data_processing_bronze_table

importlib.reload(utils.data_processing_bronze_table)

# bronze - clickstream
bronze_clickstream_directory = "datamart/bronze/clickstream/"

if not os.path.exists(bronze_clickstream_directory):
    os.makedirs(bronze_clickstream_directory)

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"
dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_clickstream_table(
        date_str, 
        bronze_clickstream_directory, 
        spark
    )

# SILVER (working)

## EDA

In [ ]:
clicks_df = spark.read.csv('data/feature_clickstream.csv', header = True, inferSchema = True)
attr_df = spark.read.csv('data/features_attributes.csv', header = True, inferSchema = True)
fin_df = spark.read.csv('data/features_financials.csv', header = True, inferSchema = True)

In [ ]:
from pyspark.sql.functions import col, sum, isnan, when
def spark_info(df):
    print(f"Rows: {df.count()}, Columns: {len(df.columns)}")
    print(f"\nSchema:")
    df.printSchema()
    print(f"Null counts:")
    df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show()


### Features_attributes

In [ ]:
spark_info(attr_df)

print(f'\nDescribe table:')
print(attr_df.describe().show())

In [ ]:
from pyspark.sql.functions import countDistinct

attr_df.select(
    countDistinct("Customer_ID").alias("custID_uinque"),
    countDistinct("Occupation").alias("occu_unique"),
    countDistinct("SSN").alias("ssn_unique")
).show()

In [ ]:
attr_df.groupBy("Occupation").count().orderBy("count", ascending = False).show()

In [ ]:
attr_df.groupBy("SSN").count().orderBy("count", ascending = False).show(5)

In [ ]:
# Convert just the one column to pandas
age_pd = attr_df.select("Age").toPandas()

# convert datatype
age_pd["Age"] = age_pd["Age"].str.strip().str.replace("_", "").astype(int)


# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(age_pd["Age"], bins=40, edgecolor="black")
plt.title("Distribution of Age")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
age_pd[age_pd['Age'] > 100].sort_values('Age')

In [ ]:
attr_df[attr_df['Occupation'] == '_______'].show()

In [5]:
def process_silver_attr(bronze_directory, silver_directory, spark):
    # connect to bronze table
    filepath = bronze_directory
    df = spark.read.csv(filepath, header = True, inferSchema = True)


    # remove trailing "_" in age
    df = df.withColumn("Age", F.regexp_replace(col("Age"), "_$", ""))

    # enforce schema
    column_type_map = {
        "Customer_ID": StringType(),
        "Name": StringType(), 
        "Age": IntegerType(), 
        "SSN": StringType(), 
        "Occupation": StringType(),
        "snapshot_date": DateType()
    }

    for column, new_type in column_type_map.items():
        df = df.withColumn(column, col(column).cast(new_type))

    # replace invalid SSN with null
    df = df.withColumn(
        "SSN",
        F.when(col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), col("SSN")).otherwise(None)
    )

    # replace invalid age with null
    df = df.withColumn(
        "Age",
        F.when((col("Age") > 0) & (col("Age") <= 120), col("Age")).otherwise(None)
    ) 

    # replace invalid occupation with null
    df = df.withColumn(
        "Occupation",
        F.when(col("Occupation") == '_______', None).otherwise(col("Occupation"))
    )


    # # create directory path
    # if not os.path.exists(silver_directory):
    #     os.makedirs(silver_directory)
    
    # # save to datamart OR JOIN WITH FINANCIALS????
    # filepath = silver_directory + 'silver_cust_attr.csv'
    # df.toPandas().to_csv(filepath, index = False)
    # print("saved to: ", filepath)

    return df



In [ ]:
temp = process_silver_attr('data/features_attributes.csv', 'datamart/silver/cust_attr/', spark)

In [ ]:
spark_info(temp)

print(f'\nDescribe table:')
print(temp.describe().show())

In [ ]:
# temp = process_silver_attr(
#     'datamart/bronze/attributes/bronze_cust_attr.csv',
#     'datamart/silver/silver_cust_attr.csv', 
#     spark)

### Features_financials

In [ ]:
spark_info(fin_df)

print(f'\nDescribe table:')
print(fin_df.describe().show())

In [27]:
def count_loan_type(entry, loan_type):
    if entry is None:
        return 0
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    return loans.count(loan_type)

def process_silver_fin(bronze_directory, silver_directory, spark):
    # connect to bronze table
    df = spark.read.csv('datamart/bronze/financials/bronze_cust_fin.csv', header = True, inferSchema = True)
    
    
    # remove trailing "_" in specified columns
    cols_to_clean = ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", 
                    "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly",
                    "Monthly_Balance"]
    
    df = df.select(
        [
            F.regexp_replace(col(c), r"^_+|_+$", "").alias(c)
            if c in cols_to_clean
            else col(c)
            for c in df.columns
        ]
    )
    
    
    # convert Credit_History_Age to integer (years)
    df = df.withColumn("years", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Year", 1).cast("integer")) \
           .withColumn("months", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Month", 1).cast("integer")) \
           .withColumn("Credit_History_Age_Years", F.round(col("years") + col("months") / 12, 2)) \
           .drop("years", "months", "Credit_History_Age")
    
    
    # enforce schema
    column_type_map = {
        'Customer_ID': StringType(),
        'Annual_Income': FloatType(), 
        'Monthly_Inhand_Salary': FloatType(), 
        'Num_Bank_Accounts': IntegerType(), 
        'Num_Credit_Card': IntegerType(),
        'Interest_Rate': FloatType(),
        'Num_of_Loan': IntegerType(), 
        'Type_of_Loan': StringType(), 
        'Delay_from_due_date': IntegerType(), 
        'Num_of_Delayed_Payment': IntegerType(), 
        'Changed_Credit_Limit': FloatType(), 
        'Num_Credit_Inquiries': IntegerType(), 
        'Credit_Mix': StringType(), 
        'Outstanding_Debt': FloatType(),
        'Credit_Utilization_Ratio': FloatType(),
        'Credit_History_Age_Years': FloatType(), 
        'Payment_of_Min_Amount': StringType(), 
        'Total_EMI_per_month': FloatType(),
        'Amount_invested_monthly': FloatType(),
        'Payment_Behaviour': StringType(),
        'Monthly_Balance': FloatType(),
        'snapshot_date': DateType()
    }
    
    for column, new_type in column_type_map.items():
        df = df.withColumn(column, col(column).cast(new_type))
    
    
    # handle anomalous values for specified quantitative columns
    cols_iqr = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Delay_from_due_date', 
                'Num_of_Delayed_Payment', 'Num_Credit_Inquiries']
    
    for c in cols_iqr:
        df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))
        q1 = df.approxQuantile(c, [0.25], 0.0)[0]
        q3 = df.approxQuantile(c, [0.75], 0.0)[0]
        iqr = q3 - q1
        upper = q3 + 1.5 * iqr
        df = df.withColumn(
            c,
            F.when((col(c) >= 0) & (col(c) <= upper), col(c)).otherwise(None)
        )

    # handle negative values for other quantitative columns
    cols_others = ['Annual_Income', 'Interest_Rate', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
                   'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

    for c in cols_others:
        df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))


    # replace Type_of_Loan with columns specifying type of loan and the count
    ## 1. get unique loan types 
    loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]
    
    all_loans = []
    for entry in loan_series:
        cleaned = re.sub(r",\s+and\s+", ", ", entry)
        loans = [loan.strip() for loan in cleaned.split(", ")]
        all_loans.extend(loans)
    
    unique_loans = sorted(set(all_loans))
    
    ## 2. add a column for each loan type
    for loan_type in unique_loans:
        col_name = "Loan_" + loan_type.replace(" ", "_").replace("-", "_")
        
        count_udf = udf(lambda x: count_loan_type(x, loan_type), IntegerType())
        
        df = df.withColumn(col_name, count_udf(col("Type_of_Loan")))
    
    ## 3. drop original column
    df = df.drop("Type_of_Loan")
    
    
    # replace "_" in Credit_Mix 
    df = df.withColumn("Credit_Mix",
        F.when(col("Credit_Mix") == "_", None).otherwise(col("Credit_Mix"))
    )
    
    # separate Payment_Behaviour into 2 columns
    valid_pattern = r"^[A-Za-z]+_spent_[A-Za-z]+_value_payments$"
    
    df = (
        df
        .withColumn("parts", F.split(col("Payment_Behaviour"), "_"))
        .withColumn("Spending_Behaviour",
                    F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[0]).otherwise(None)
                   )
        .withColumn("Payments_Size",
                    F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[2]).otherwise(None)
                   )
        .drop("Payment_Behaviour", "parts")
    )

    return df
    

In [ ]:
def count_loan_type(entry, loan_type):
    if entry is None:
        return 0
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    return loans.count(loan_type)


    
# connect to bronze table
df = spark.read.csv('datamart/bronze/financials/bronze_cust_fin.csv', header = True, inferSchema = True)


# remove trailing "_" in specified columns
cols_to_clean = ["Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment", 
                "Changed_Credit_Limit", "Outstanding_Debt"]

df = df.select(
    [F.regexp_replace(col(c), "_$", "").alias(c) if c in cols_to_clean 
     else col(c) 
     for c in df.columns]
)


# convert Credit_History_Age to integer (years)
df = df.withColumn("years", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Year", 1).cast("integer")) \
       .withColumn("months", F.regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Month", 1).cast("integer")) \
       .withColumn("Credit_History_Age_Years", F.round(col("years") + col("months") / 12, 2)) \
       .drop("years", "months", "Credit_History_Age")


# enforce schema
column_type_map = {
    'Customer_ID': StringType(),
    'Annual_Income': FloatType(), 
    'Monthly_Inhand_Salary': FloatType(), 
    'Num_Bank_Accounts': IntegerType(), 
    'Num_Credit_Card': IntegerType(),
    'Interest_Rate': FloatType(),
    'Num_of_Loan': IntegerType(), 
    'Type_of_Loan': StringType(), 
    'Delay_from_due_date': IntegerType(), 
    'Num_of_Delayed_Payment': IntegerType(), 
    'Changed_Credit_Limit': FloatType(), 
    'Num_Credit_Inquiries': IntegerType(), 
    'Credit_Mix': StringType(), 
    'Outstanding_Debt': FloatType(),
    'Credit_Utilization_Ratio': FloatType(),
    'Credit_History_Age_Years': FloatType(), 
    'Payment_of_Min_Amount': StringType(), 
    'Total_EMI_per_month': FloatType(),
    'Amount_invested_monthly': FloatType(),
    'Payment_Behaviour': StringType(),
    'Monthly_Balance': FloatType(),
    'snapshot_date': DateType()
}

for column, new_type in column_type_map.items():
    df = df.withColumn(column, col(column).cast(new_type))


# handle anomalous values for specified quantitative columns
cols_iqr = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 'Delay_from_due_date', 
            'Num_of_Delayed_Payment', 'Num_Credit_Inquiries']

for c in cols_iqr:
    df = df.withColumn(c, F.when(col(c)<0, None).otherwise(col(c)))
    q1 = df.approxQuantile(c, [0.25], 0.0)[0]
    q3 = df.approxQuantile(c, [0.75], 0.0)[0]
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    df = df.withColumn(
        c,
        when((col(c) >= 0) & (col(c) <= upper), col(c)).otherwise(None)
    )


# replace Type_of_Loan with columns specifying type of loan and the count
## 1. get unique loan types 
loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]

all_loans = []
for entry in loan_series:
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

unique_loans = sorted(set(all_loans))

## 2. add a column for each loan type
for loan_type in unique_loans:
    col_name = "Loan_" + loan_type.replace(" ", "_").replace("-", "_")
    
    count_udf = udf(lambda x: count_loan_type(x, loan_type), IntegerType())
    
    df = df.withColumn(col_name, count_udf(col("Type_of_Loan")))

## 3. drop original column
df = df.drop("Type_of_Loan")


# replace "_" in Credit_Mix 
df = df.withColumn("Credit_Mix",
    F.when(col("Credit_Mix") == "_", None).otherwise(col("Credit_Mix"))
)

# separate Payment_Behaviour into 2 columns
valid_pattern = r"^[A-Za-z]+_spent_[A-Za-z]+_value_payments$"

df = (
    df
    .withColumn("parts", F.split(col("Payment_Behaviour"), "_"))
    .withColumn("Spending_Behaviour",
                F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[0]).otherwise(None)
               )
    .withColumn("Payments_Size",
                F.when(col("Payment_Behaviour").rlike(valid_pattern), col("parts")[2]).otherwise(None)
               )
    .drop("Payment_Behaviour", "parts")
)

In [ ]:

df.toPandas().describe(include = 'object')

In [ ]:
df_pd['Payment_Behaviour'].value_counts()

In [ ]:
import re


# Step 1 — get unique loan types (use pandas for this)
loan_series = df.select("Type_of_Loan").dropna().toPandas()["Type_of_Loan"]

all_loans = []
for entry in loan_series:
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

unique_loans = sorted(set(all_loans))
print(unique_loans)



In [ ]:
df_pd = df.toPandas()
df_pd.describe(include = 'object')

In [ ]:
import re

# Collect all loan values into a list
loan_series = df_pd["Type_of_Loan"].dropna()

# Split each row and clean up
all_loans = []
for entry in loan_series:
    # Remove " and " before last item, then split by ", "
    cleaned = re.sub(r",\s+and\s+", ", ", entry)
    loans = [loan.strip() for loan in cleaned.split(", ")]
    all_loans.extend(loans)

# Get unique values
unique_loans = set(all_loans)
print(f"Unique loan types ({len(unique_loans)}):")
for loan in sorted(unique_loans):
    print(f"  - {loan}")

In [ ]:
temp = df_pd.copy()
temp["Total_EMI_per_month"] = temp["Total_EMI_per_month"].where(temp["Total_EMI_per_month"] >= 0, None)

# Calculate IQR
Q1 = temp["Total_EMI_per_month"].quantile(0.25)
Q3 = temp["Total_EMI_per_month"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower: {lower}")
print(f"Upper: {upper}")

# Delay_from_due_date	Num_of_Delayed_Payment	

In [ ]:
df_pd[df_pd["Num_Credit_Inquiries"]>19].sort_values("Num_Credit_Inquiries")

In [ ]:
# Convert to pandas
credit_card_pd = df.select("Total_EMI_per_month").toPandas()

# Sort values and calculate cumulative percentage
sorted_vals = np.sort(credit_card_pd["Total_EMI_per_month"])
cumulative = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals) * 100

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sorted_vals, cumulative, marker=".", linestyle="-")
plt.title("Cumulative Distribution of Num_of_Loan")
plt.xlabel("Num_of_Loan")
plt.ylabel("Cumulative %")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df_pd["Total_EMI_per_month"].describe()
df_pd["Total_EMI_per_month"].value_counts().sort_index()

# Plot to see distribution
df_pd["Total_EMI_per_month"].hist(bins=30)

In [ ]:
df_pd["Delay_from_due_date"].describe()
df_pd["Delay_from_due_date"].value_counts().sort_index()

# Plot to see distribution
df_pd["Delay_from_due_date"].hist(bins=30)

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
df_pd = df.toPandas()
df_pd.describe()

In [ ]:
df_pd.describe(include = 'object')

### Check attr and fin????

In [9]:
attr_silver = process_silver_attr('datamart/bronze/attributes/bronze_cust_attr.csv', 
                                  'datamart/silver/attributes/', 
                                  spark)

attr_silver.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer_ID    12500 non-null  object 
 1   Name           12500 non-null  object 
 2   Age            12181 non-null  float64
 3   SSN            11797 non-null  object 
 4   Occupation     11620 non-null  object 
 5   snapshot_date  12500 non-null  object 
dtypes: float64(1), object(5)
memory usage: 586.1+ KB


In [28]:
fin_silver = process_silver_fin('datamart/bronze/financials/bronze_cust_fin.csv',
                                'datamart/silver/financials/',
                                spark)

fin_silver.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   12500 non-null  object 
 1   Annual_Income                 12500 non-null  float32
 2   Monthly_Inhand_Salary         12500 non-null  float32
 3   Num_Bank_Accounts             12329 non-null  float64
 4   Num_Credit_Card               12204 non-null  float64
 5   Interest_Rate                 12500 non-null  float32
 6   Num_of_Loan                   11933 non-null  float64
 7   Delay_from_due_date           11908 non-null  float64
 8   Num_of_Delayed_Payment        12311 non-null  float64
 9   Changed_Credit_Limit          12246 non-null  float32
 10  Num_Credit_Inquiries          12305 non-null  float64
 11  Credit_Mix                    9889 non-null   object 
 12  Outstanding_Debt              12500 non-null  float32
 13  C

### LMS

In [ ]:
lms_df = spark.read.csv('data/lms_loan_daily.csv', header = True, inferSchema = True)

In [ ]:
df_pd = lms_df.toPandas()

In [ ]:
df_pd.info()

In [ ]:
df_pd.describe()

In [ ]:
df_pd.describe(include = 'object')